<a href="https://colab.research.google.com/github/semmatoninn/CHE1148_MOFs_AtomicWizards/blob/main/XGBoostBaseline_MOFs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# STEP 1 — Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
from pathlib import Path
import torch
import os

BASE_DIR = Path("/content/drive/MyDrive/MOF_Conf/processed_pyg")

train_dir = BASE_DIR / "train"
val_dir   = BASE_DIR / "val"
test_dir  = BASE_DIR / "test"

print("Train exists:", train_dir.exists())
print("Val exists:", val_dir.exists())
print("Test exists:", test_dir.exists())

Train exists: True
Val exists: True
Test exists: True


In [4]:
train_files = sorted(train_dir.glob("*.pt"))

print("Number of train files:", len(train_files))
print("First 5 files:", train_files[:5])

Number of train files: 4
First 5 files: [PosixPath('/content/drive/MyDrive/MOF_Conf/processed_pyg/train/train_part_000.pt'), PosixPath('/content/drive/MyDrive/MOF_Conf/processed_pyg/train/train_part_001.pt'), PosixPath('/content/drive/MyDrive/MOF_Conf/processed_pyg/train/train_part_002.pt'), PosixPath('/content/drive/MyDrive/MOF_Conf/processed_pyg/train/train_part_003.pt')]


In [5]:
# Only run if needed
!pip install torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 63.5 MB/s eta 0:00:00


In [6]:
import torch
from torch_geometric.data import Data

In [7]:
import torch

# make sure file exists
print(train_files[0])

# load with correct flag (important for PyTorch 2.6+)
sample_data = torch.load(train_files[0], map_location="cpu", weights_only=False)

print("Loaded successfully ✅")
print("Type:", type(sample_data))
print("Length:", len(sample_data))

/content/drive/MyDrive/MOF_Conf/processed_pyg/train/train_part_000.pt
Loaded successfully ✅
Type: <class 'list'>
Length: 500


In [8]:
g = sample_data[0]

print("Graph object:")
print(g)

print("\nAttributes:")
for attr in ["x", "z", "pos", "edge_index", "edge_attr", "y", "mof_name", "sample_name", "cell", "pbc"]:
    print(f"{attr}: {hasattr(g, attr)}")
    if hasattr(g, attr):
        val = getattr(g, attr)
        try:
            print("   shape:", val.shape)
        except:
            print("   value:", val)

Graph object:
Data(y=[1], pos=[277, 3], z=[277], cell=[3, 3], pbc=[3], mof_name='ADOBEB_charged', config_name='ADOBEB_charged_w_CO2_1', num_atoms=277, split='train')

Attributes:
x: True
   value: None
z: True
   shape: torch.Size([277])
pos: True
   shape: torch.Size([277, 3])
edge_index: True
   value: None
edge_attr: True
   value: None
y: True
   shape: torch.Size([1])
mof_name: True
   value: ADOBEB_charged
sample_name: False
cell: True
   shape: torch.Size([3, 3])
pbc: True
   shape: torch.Size([3])


In [9]:
# STEP 7 — Handcrafted feature extraction for one graph

import numpy as np

METAL_Z = {
    3, 4, 11, 12, 13, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30,
    31, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50,
    55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
    72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83
}

def cell_volume(cell):
    a, b, c = cell[0], cell[1], cell[2]
    return abs(np.dot(a, np.cross(b, c)))

def extract_features(g):
    z = g.z.cpu().numpy()
    pos = g.pos.cpu().numpy()
    cell = g.cell.cpu().numpy() if hasattr(g, "cell") and g.cell is not None else None

    feats = {}

    # -------------------------
    # basic size / composition
    # -------------------------
    feats["num_atoms"] = len(z)
    feats["mean_z"] = float(np.mean(z))
    feats["std_z"] = float(np.std(z))
    feats["min_z"] = int(np.min(z))
    feats["max_z"] = int(np.max(z))

    # -------------------------
    # element counts
    # -------------------------
    feats["count_H"] = int(np.sum(z == 1))
    feats["count_C"] = int(np.sum(z == 6))
    feats["count_N"] = int(np.sum(z == 7))
    feats["count_O"] = int(np.sum(z == 8))
    feats["count_F"] = int(np.sum(z == 9))
    feats["count_S"] = int(np.sum(z == 16))

    # -------------------------
    # element fractions
    # -------------------------
    feats["frac_H"] = float(np.mean(z == 1))
    feats["frac_C"] = float(np.mean(z == 6))
    feats["frac_N"] = float(np.mean(z == 7))
    feats["frac_O"] = float(np.mean(z == 8))

    # -------------------------
    # metal information
    # -------------------------
    is_metal = np.isin(z, list(METAL_Z))
    feats["num_metal_atoms"] = int(np.sum(is_metal))
    feats["metal_fraction"] = float(np.mean(is_metal))

    # -------------------------
    # diversity
    # -------------------------
    feats["num_unique_elements"] = int(len(np.unique(z)))

    # -------------------------
    # coordinate geometry
    # -------------------------
    center = np.mean(pos, axis=0)
    dists_center = np.linalg.norm(pos - center, axis=1)
    feats["mean_dist_to_center"] = float(np.mean(dists_center))
    feats["std_dist_to_center"] = float(np.std(dists_center))
    feats["max_dist_to_center"] = float(np.max(dists_center))

    # spatial spread
    feats["range_x"] = float(np.max(pos[:, 0]) - np.min(pos[:, 0]))
    feats["range_y"] = float(np.max(pos[:, 1]) - np.min(pos[:, 1]))
    feats["range_z"] = float(np.max(pos[:, 2]) - np.min(pos[:, 2]))

    # -------------------------
    # cell features
    # -------------------------
    if cell is not None:
        a_len = np.linalg.norm(cell[0])
        b_len = np.linalg.norm(cell[1])
        c_len = np.linalg.norm(cell[2])
        vol = cell_volume(cell)

        feats["cell_a"] = float(a_len)
        feats["cell_b"] = float(b_len)
        feats["cell_c"] = float(c_len)
        feats["cell_volume"] = float(vol)
        feats["atom_density"] = float(len(z) / vol) if vol > 0 else 0.0
    else:
        feats["cell_a"] = 0.0
        feats["cell_b"] = 0.0
        feats["cell_c"] = 0.0
        feats["cell_volume"] = 0.0
        feats["atom_density"] = 0.0

    # -------------------------
    # pairwise distance features
    # -------------------------
    if len(pos) > 300:
        idx = np.random.choice(len(pos), size=300, replace=False)
        pos_sample = pos[idx]
    else:
        pos_sample = pos

    pairwise = np.linalg.norm(
        pos_sample[:, None, :] - pos_sample[None, :, :],
        axis=-1
    )
    pairwise = pairwise[np.triu_indices_from(pairwise, k=1)]

    if len(pairwise) > 0:
        feats["dist_mean"] = float(np.mean(pairwise))
        feats["dist_std"] = float(np.std(pairwise))
        feats["dist_min"] = float(np.min(pairwise))
        feats["dist_max"] = float(np.max(pairwise))
    else:
        feats["dist_mean"] = 0.0
        feats["dist_std"] = 0.0
        feats["dist_min"] = 0.0
        feats["dist_max"] = 0.0

    # -------------------------
    # metal / nonmetal distance proxy
    # -------------------------
    metal_mask = np.isin(z, list(METAL_Z))
    nonmetal_mask = ~metal_mask

    if np.any(metal_mask) and np.any(nonmetal_mask):
        metal_pos = pos[metal_mask]
        nonmetal_pos = pos[nonmetal_mask]

        mn_dists = np.linalg.norm(
            metal_pos[:, None, :] - nonmetal_pos[None, :, :],
            axis=-1
        )
        feats["metal_nonmetal_dist_mean"] = float(np.mean(mn_dists))
        feats["metal_nonmetal_dist_min"] = float(np.min(mn_dists))
    else:
        feats["metal_nonmetal_dist_mean"] = 0.0
        feats["metal_nonmetal_dist_min"] = 0.0

    # -------------------------
    # near-center proxy features
    # -------------------------
    threshold = np.percentile(dists_center, 20)
    near_mask = dists_center <= threshold

    if np.any(near_mask):
        feats["near_center_frac_metal"] = float(np.mean(np.isin(z[near_mask], list(METAL_Z))))
        feats["near_center_density"] = float(np.sum(near_mask) / len(z))
        feats["near_center_mean_z"] = float(np.mean(z[near_mask]))
    else:
        feats["near_center_frac_metal"] = 0.0
        feats["near_center_density"] = 0.0
        feats["near_center_mean_z"] = 0.0

    return feats

In [10]:
feat_dict = extract_features(g)
print("Number of features:", len(feat_dict))
for k, v in feat_dict.items():
    print(k, ":", v)

Number of features: 38
num_atoms : 277
mean_z : 7.783393501805054
std_z : 6.576680060190657
min_z : 1
max_z : 25
count_H : 64
count_C : 81
count_N : 16
count_O : 78
count_F : 0
count_S : 0
frac_H : 0.23104693140794225
frac_C : 0.2924187725631769
frac_N : 0.05776173285198556
frac_O : 0.2815884476534296
num_metal_atoms : 36
metal_fraction : 0.1299638989169675
num_unique_elements : 7
mean_dist_to_center : 7.630995273590088
std_dist_to_center : 2.430555820465088
max_dist_to_center : 13.95782470703125
range_x : 14.600675582885742
range_y : 16.909826278686523
range_z : 16.754697799682617
cell_a : 14.73765754699707
cell_b : 16.910051345825195
cell_c : 16.9100341796875
cell_volume : 4214.2265625
atom_density : 0.06572973728179932
dist_mean : 10.591949462890625
dist_std : 4.068753719329834
dist_min : 1.0849707126617432
dist_max : 25.349260330200195
metal_nonmetal_dist_mean : 10.551504135131836
metal_nonmetal_dist_min : 1.6210076808929443
near_center_frac_metal : 0.2857142857142857
near_center_d

In [11]:
# STEP 9 — Convert one split folder into a tabular dataframe

import pandas as pd

def load_split_as_dataframe(split_files):
    rows = []

    for i, file_path in enumerate(split_files):
        print(f"Loading {i+1}/{len(split_files)}: {file_path.name}")
        data_list = torch.load(file_path, map_location="cpu", weights_only=False)

        for g in data_list:
            feats = extract_features(g)

            feats["target"] = float(g.y.item())
            feats["mof_name"] = g.mof_name if hasattr(g, "mof_name") else None
            feats["config_name"] = g.config_name if hasattr(g, "config_name") else None
            feats["split"] = g.split if hasattr(g, "split") else None

            rows.append(feats)

    df = pd.DataFrame(rows)
    return df

In [14]:
# STEP 11 — List val and test files

val_files = sorted(val_dir.glob("*.pt"))
test_files = sorted(test_dir.glob("*.pt"))

print("Val files:", len(val_files))
print("Test files:", len(test_files))
print("First val file:", val_files[:1])
print("First test file:", test_files[:1])

Val files: 1
Test files: 1
First val file: [PosixPath('/content/drive/MyDrive/MOF_Conf/processed_pyg/val/val_part_000.pt')]
First test file: [PosixPath('/content/drive/MyDrive/MOF_Conf/processed_pyg/test/test_part_000.pt')]


In [15]:
train_df = load_split_as_dataframe(train_files)
val_df   = load_split_as_dataframe(val_files)
test_df  = load_split_as_dataframe(test_files)

print(train_df.shape, val_df.shape, test_df.shape)

Loading 1/4: train_part_000.pt
Loading 2/4: train_part_001.pt
Loading 3/4: train_part_002.pt
Loading 4/4: train_part_003.pt
Loading 1/1: val_part_000.pt
Loading 1/1: test_part_000.pt
(1860, 42) (427, 42) (444, 42)


In [16]:
# STEP 11 — Check columns and missing values

print(train_df.columns.tolist())
print("\nMissing values per column:")
print(train_df.isna().sum())

print("\nVal missing values per column:")
print(val_df.isna().sum())

print("\nTest missing values per column:")
print(test_df.isna().sum())

['num_atoms', 'mean_z', 'std_z', 'min_z', 'max_z', 'count_H', 'count_C', 'count_N', 'count_O', 'count_F', 'count_S', 'frac_H', 'frac_C', 'frac_N', 'frac_O', 'num_metal_atoms', 'metal_fraction', 'num_unique_elements', 'mean_dist_to_center', 'std_dist_to_center', 'max_dist_to_center', 'range_x', 'range_y', 'range_z', 'cell_a', 'cell_b', 'cell_c', 'cell_volume', 'atom_density', 'dist_mean', 'dist_std', 'dist_min', 'dist_max', 'metal_nonmetal_dist_mean', 'metal_nonmetal_dist_min', 'near_center_frac_metal', 'near_center_density', 'near_center_mean_z', 'target', 'mof_name', 'config_name', 'split']

Missing values per column:
num_atoms                   0
mean_z                      0
std_z                       0
min_z                       0
max_z                       0
count_H                     0
count_C                     0
count_N                     0
count_O                     0
count_F                     0
count_S                     0
frac_H                      0
frac_C       

In [17]:
# STEP 13 — Build validation dataframe

val_df = load_split_as_dataframe(val_files)

print("Val shape:", val_df.shape)
print(val_df.head())

Loading 1/1: val_part_000.pt
Val shape: (427, 42)
   num_atoms    mean_z     std_z  min_z  max_z  count_H  count_C  count_N  \
0        333  6.144144  7.796411      1     48       96      169        8   
1        333  6.144144  7.796411      1     48       96      169        8   
2        339  6.094395  7.748438      1     48      100      169        8   
3        339  6.094395  7.748438      1     48      100      169        8   
4        339  6.094395  7.748438      1     48      100      169        8   

   count_O  count_F  ...   dist_max  metal_nonmetal_dist_mean  \
0       50        0  ...  30.055618                 12.311837   
1       50        0  ...  30.215481                 12.318774   
2       52        0  ...  30.080643                 12.302788   
3       52        0  ...  30.050198                 12.277911   
4       52        0  ...  30.018393                 12.227077   

   metal_nonmetal_dist_min  near_center_frac_metal  near_center_density  \
0                 2.2

In [18]:
# STEP 14 — Build test dataframe

test_df = load_split_as_dataframe(test_files)

print("Test shape:", test_df.shape)
print(test_df.head())

Loading 1/1: test_part_000.pt
Test shape: (444, 42)
   num_atoms    mean_z     std_z  min_z  max_z  count_H  count_C  count_N  \
0        387  5.782946  5.709543      1     30      125      157       21   
1        387  5.782946  5.709543      1     30      125      157       21   
2        393  5.745547  5.688378      1     30      129      157       21   
3        393  5.745547  5.688378      1     30      129      157       21   
4        393  5.745547  5.688378      1     30      129      157       21   

   count_O  count_F  ...   dist_max  metal_nonmetal_dist_mean  \
0       68        0  ...  49.111919                 17.537868   
1       68        0  ...  49.137074                 17.551895   
2       70        0  ...  50.953545                 17.557034   
3       70        0  ...  50.946068                 17.504332   
4       70        0  ...  49.126007                 17.533165   

   metal_nonmetal_dist_min  near_center_frac_metal  near_center_density  \
0                 1

In [19]:
# STEP 15 — Sanity checks

print("Train split labels:", train_df["split"].value_counts(dropna=False).to_dict())
print("Val split labels:", val_df["split"].value_counts(dropna=False).to_dict())
print("Test split labels:", test_df["split"].value_counts(dropna=False).to_dict())

print("Unique train MOFs:", train_df["mof_name"].nunique())
print("Unique val MOFs:", val_df["mof_name"].nunique())
print("Unique test MOFs:", test_df["mof_name"].nunique())

Train split labels: {'train': 1860}
Val split labels: {'val': 427}
Test split labels: {'test': 444}
Unique train MOFs: 135
Unique val MOFs: 29
Unique test MOFs: 30


In [20]:
# Columns to use as X
feature_cols = [
    c for c in train_df.columns
    if c not in ["target", "mof_name", "config_name", "split"]
]

print("Number of feature columns:", len(feature_cols))
print(feature_cols)

Number of feature columns: 38
['num_atoms', 'mean_z', 'std_z', 'min_z', 'max_z', 'count_H', 'count_C', 'count_N', 'count_O', 'count_F', 'count_S', 'frac_H', 'frac_C', 'frac_N', 'frac_O', 'num_metal_atoms', 'metal_fraction', 'num_unique_elements', 'mean_dist_to_center', 'std_dist_to_center', 'max_dist_to_center', 'range_x', 'range_y', 'range_z', 'cell_a', 'cell_b', 'cell_c', 'cell_volume', 'atom_density', 'dist_mean', 'dist_std', 'dist_min', 'dist_max', 'metal_nonmetal_dist_mean', 'metal_nonmetal_dist_min', 'near_center_frac_metal', 'near_center_density', 'near_center_mean_z']


In [21]:
# STEP 12 — Build X / y

feature_cols = [c for c in train_df.columns if c not in ["target", "mof_name", "config_name", "split"]]

X_train = train_df[feature_cols].copy()
y_train = train_df["target"].copy()

X_val = val_df[feature_cols].copy()
y_val = val_df["target"].copy()

X_test = test_df[feature_cols].copy()
y_test = test_df["target"].copy()

print("Num features:", len(feature_cols))
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

Num features: 38
X_train: (1860, 38)
X_val: (427, 38)
X_test: (444, 38)


In [22]:
!pip -q install xgboost

In [23]:
import xgboost as xgb
import numpy as np

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr, kendalltau


In [24]:
np.random.seed(42)

model = xgb.XGBRegressor(
    n_estimators=1000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=30
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=50
)


[0]	validation_0-rmse:0.59360
[50]	validation_0-rmse:0.53761
[98]	validation_0-rmse:0.53692


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=30,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=1000,
             n_jobs=-1, num_parallel_tree=None, ...)

In [25]:
# STEP 14 — Predict

train_pred = model.predict(X_train)
val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

In [26]:
# STEP 15 — Metrics

def regression_metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    sp = spearmanr(y_true, y_pred).statistic
    kd = kendalltau(y_true, y_pred).statistic

    return {
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae,
        "Spearman": sp,
        "Kendall": kd
    }

print("Train row-level:", regression_metrics(y_train, train_pred))
print("Val row-level:", regression_metrics(y_val, val_pred))
print("Test row-level:", regression_metrics(y_test, test_pred))

Train row-level: {'R2': 0.7924326087392805, 'RMSE': np.float64(0.2294401225020479), 'MAE': 0.1696906927482335, 'Spearman': np.float64(0.884645116962902), 'Kendall': np.float64(0.7084483837649035)}
Val row-level: {'R2': 0.19503424099427924, 'RMSE': np.float64(0.5307041139795148), 'MAE': 0.3413923779814406, 'Spearman': np.float64(0.5181729375194647), 'Kendall': np.float64(0.3572229526085293)}
Test row-level: {'R2': 0.20088307178926434, 'RMSE': np.float64(0.4653353820675052), 'MAE': 0.34329243467899695, 'Spearman': np.float64(0.5267159336277837), 'Kendall': np.float64(0.3488382695397352)}


In [27]:
# STEP 16 — MOF-level aggregation

train_df["pred"] = train_pred
val_df["pred"] = val_pred
test_df["pred"] = test_pred

def aggregate_mof(df, agg="mean"):
    if agg == "mean":
        out = df.groupby("mof_name", as_index=False).agg({
            "target": "mean",
            "pred": "mean"
        })
    elif agg == "min":
        out = df.groupby("mof_name", as_index=False).agg({
            "target": "min",
            "pred": "min"
        })
    else:
        raise ValueError("agg must be 'mean' or 'min'")
    return out

test_mof_mean = aggregate_mof(test_df, agg="mean")
test_mof_min = aggregate_mof(test_df, agg="min")

print("Test MOF-level (mean):", regression_metrics(test_mof_mean["target"], test_mof_mean["pred"]))
print("Test MOF-level (min):", regression_metrics(test_mof_min["target"], test_mof_min["pred"]))

Test MOF-level (mean): {'R2': 0.19164271859123239, 'RMSE': np.float64(0.26004111814577663), 'MAE': 0.20390932607712028, 'Spearman': np.float64(0.4972191323692992), 'Kendall': np.float64(0.31494252873563217)}
Test MOF-level (min): {'R2': -0.39714690093082705, 'RMSE': np.float64(0.7710677211369257), 'MAE': 0.5611088797450066, 'Spearman': np.float64(0.578642936596218), 'Kendall': np.float64(0.40689655172413797)}


In [28]:
def top_k_recall(mof_df, k, lower_is_better=True):
    """
    mof_df must contain columns:
      - mof_name
      - target
      - pred
    """
    if k <= 0:
        raise ValueError("k must be positive")
    if k > len(mof_df):
        raise ValueError(f"k={k} is larger than number of MOFs ({len(mof_df)})")

    ascending = lower_is_better

    true_top = (
        mof_df.sort_values("target", ascending=ascending)
        .head(k)["mof_name"]
        .tolist()
    )

    pred_top = (
        mof_df.sort_values("pred", ascending=ascending)
        .head(k)["mof_name"]
        .tolist()
    )

    true_top_set = set(true_top)
    pred_top_set = set(pred_top)

    hits = len(true_top_set & pred_top_set)
    recall = hits / k

    return {
        "k": k,
        "hits": hits,
        "recall": recall,
        "true_top": true_top,
        "pred_top": pred_top
    }

In [29]:
# mean
top_k_recall(test_mof_mean, k=10)

# min
top_k_recall(test_mof_min, k=10)

{'k': 10,
 'hits': 5,
 'recall': 0.5,
 'true_top': ['TIXCIO',
  'FIJZEE_0.04_0',
  'TUMQID_0.12_0',
  'DAGDIZ',
  'ATOWIQ_0.16_0',
  'VUHJAK',
  'XEHSIN_0.02_0',
  'HAHPIQ_charged_0.08_0',
  'VIWZOR',
  'ATOWIQ_0.16_1'],
 'pred_top': ['YEGKOM',
  'XEHSIN_0.02_0',
  'RILVOZ',
  'ja506357n_si_002_0.12_0',
  'JOFKIA',
  'VUHJAK',
  'HAHPIQ_charged',
  'DAGDIZ',
  'HAHPIQ_charged_0.08_0',
  'ATOWIQ_0.16_1']}

In [30]:
for k in [10]:
    result = top_k_recall(test_mof_mean, k=k, lower_is_better=True)
    print(f"Mean agg Top-{k} recall:", result["recall"], f"({result['hits']}/{k})")

Mean agg Top-10 recall: 0.5 (5/10)


In [31]:
for k in [10]:
    result = top_k_recall(test_mof_min, k=k, lower_is_better=True)
    print(f"Min agg Top-{k} recall:", result["recall"], f"({result['hits']}/{k})")

Min agg Top-10 recall: 0.5 (5/10)


#Confidence interval

In [32]:
# ============================================================
# BOOTSTRAP CI FOR XGBOOST MOF-LEVEL METRICS
# Add this AFTER test_mof_mean / test_mof_min are created
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr, kendalltau

N_BOOT = 2000
TOP_K = 10
SEED = 42

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def compute_top_k_recall(df, k=10, lower_is_better=True):
    if k <= 0:
        raise ValueError("k must be positive")
    k = min(k, len(df))

    ascending = lower_is_better
    true_top = set(df.sort_values("target", ascending=ascending).head(k)["mof_name"].astype(str))
    pred_top = set(df.sort_values("pred", ascending=ascending).head(k)["mof_name"].astype(str))

    return float(len(true_top & pred_top) / k)

def compute_metrics(df, k=10):
    y_true = df["target"].to_numpy(dtype=float)
    y_pred = df["pred"].to_numpy(dtype=float)

    sp = spearmanr(y_true, y_pred).correlation
    kt = kendalltau(y_true, y_pred).correlation

    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": rmse(y_true, y_pred),
        "Spearman": np.nan if sp is None else float(sp),
        "Kendall": np.nan if kt is None else float(kt),
        f"Top{k}_Recall": compute_top_k_recall(df, k=k, lower_is_better=True),
    }

def bootstrap_ci(df, n_boot=2000, k=10, seed=42):
    """
    Bootstrap at the MOF level.
    df must contain: mof_name, target, pred
    """
    rng = np.random.default_rng(seed)
    df = df.copy().reset_index(drop=True)

    required_cols = ["mof_name", "target", "pred"]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    n = len(df)
    point = compute_metrics(df, k=k)

    boot_rows = []
    for b in range(n_boot):
        idx = rng.integers(0, n, size=n)
        sample = df.iloc[idx].reset_index(drop=True)
        m = compute_metrics(sample, k=k)
        m["bootstrap_id"] = b
        boot_rows.append(m)

    boot_df = pd.DataFrame(boot_rows)

    summary_rows = []
    metric_cols = [c for c in boot_df.columns if c != "bootstrap_id"]
    for metric in metric_cols:
        vals = boot_df[metric].dropna().to_numpy(dtype=float)
        lo, hi = np.percentile(vals, [2.5, 97.5])

        summary_rows.append({
            "metric": metric,
            "point": point[metric],
            "ci_lower": lo,
            "ci_upper": hi
        })

    summary_df = pd.DataFrame(summary_rows)
    return summary_df, boot_df

def print_ci_table(summary_df, title):
    print(f"\n{title}")
    print("=" * len(title))
    for _, row in summary_df.iterrows():
        print(f"{row['metric']}: {row['point']:.3f} [{row['ci_lower']:.3f}, {row['ci_upper']:.3f}]")

# ------------------------------------------------------------
# RUN BOOTSTRAP ON XGBOOST MOF-LEVEL TABLES
# ------------------------------------------------------------
summary_mean_xgb, boot_mean_xgb = bootstrap_ci(
    test_mof_mean, n_boot=N_BOOT, k=TOP_K, seed=SEED
)

summary_min_xgb, boot_min_xgb = bootstrap_ci(
    test_mof_min, n_boot=N_BOOT, k=TOP_K, seed=SEED
)

print_ci_table(summary_mean_xgb, "XGBoost TEST (mean aggregation) - Bootstrap CI")
print_ci_table(summary_min_xgb,  "XGBoost TEST (min aggregation) - Bootstrap CI")

# OPTIONAL: SAVE TABLES IN SAME STYLE AS GNN
OUT_DIR = "/content/drive/MyDrive/MOF_Conf/xgboost_results"
import os
os.makedirs(OUT_DIR, exist_ok=True)

summary_mean_xgb.to_csv(f"{OUT_DIR}/bootstrap_ci_mean_xgboost.csv", index=False)
summary_min_xgb.to_csv(f"{OUT_DIR}/bootstrap_ci_min_xgboost.csv", index=False)

boot_mean_xgb.to_csv(f"{OUT_DIR}/bootstrap_samples_mean_xgboost.csv", index=False)
boot_min_xgb.to_csv(f"{OUT_DIR}/bootstrap_samples_min_xgboost.csv", index=False)

# also save the MOF-level tables themselves for easy comparison with EGNN
test_mof_mean.to_csv(f"{OUT_DIR}/test_mof_rankings_mean_xgboost.csv", index=False)
test_mof_min.to_csv(f"{OUT_DIR}/test_mof_rankings_min_xgboost.csv", index=False)


XGBoost TEST (mean aggregation) - Bootstrap CI
MAE: 0.204 [0.152, 0.264]
RMSE: 0.260 [0.185, 0.333]
Spearman: 0.497 [0.200, 0.695]
Kendall: 0.315 [0.109, 0.491]
Top10_Recall: 0.500 [0.100, 0.600]

XGBoost TEST (min aggregation) - Bootstrap CI
MAE: 0.561 [0.397, 0.765]
RMSE: 0.771 [0.513, 1.053]
Spearman: 0.579 [0.261, 0.791]
Kendall: 0.407 [0.170, 0.608]
Top10_Recall: 0.500 [0.100, 0.600]
